In [1]:
# ===== Cell 1: Imports + Config =====
import os, json, re, gc, copy
import torch, pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm

MODEL_NAME    = '/root/autodl-tmp/employee_code/base_model'
TRAIN_JSON    = '/root/autodl-tmp/civil_ft/data/train_sft_civil_code.json'
EVAL_JSON     = '/root/autodl-tmp/civil_ft/data/eval_civil_code_questions.json'

# Stage-1 output
S1_OUTPUT_DIR = '/root/autodl-tmp/civil_ft/checkpoints'
# Stage-2 output (format alignment per checkpoint)
S2_ROOT_DIR   = '/root/autodl-tmp/civil_ft/stage2_aligned'
# Final results
RESULTS_DIR   = '/root/autodl-tmp/civil_ft/results'

MAX_SEQ_LEN   = 1024
LORA_R = 32; LORA_ALPHA = 64; LORA_DROPOUT = 0.05
LR = 1e-4; LR_S2 = 2e-5
EPOCHS = 5.0; BS = 2; GAS = 8
SAVE_STEPS = 10; LOG_STEPS = 10

os.makedirs(S1_OUTPUT_DIR, exist_ok=True)
os.makedirs(S2_ROOT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Stage-1: ~3780 samples, ~1180 steps, save every {SAVE_STEPS}')
print(f'Stage-2: format alignment on checkpoints 280-520')
print(f'Eval: 362 civil code questions')

Stage-1: ~3780 samples, ~1180 steps, save every 10
Stage-2: format alignment on checkpoints 280-520
Eval: 362 civil code questions


In [ ]:
# ===== Cell 2: Stage-1 Knowledge Injection (train + save) =====
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True)
print(f'CUDA: {torch.cuda.is_available()}')

# Dataset
SYS = '/no_think\n你是一个专业的中国法律背诵助手。请严格根据用户要求输出法条内容，不要添加解释。'
with open(TRAIN_JSON, 'r', encoding='utf-8') as f: raw = json.load(f)
examples = []
for item in raw:
    inst = item.get('instruction','').strip()
    inp  = item.get('input','').strip()
    out  = item.get('output','').strip()
    if not inst or not out: continue
    msgs = [{'role':'system','content':SYS},{'role':'user','content':inst if not inp else f'{inst}\n\n{inp}'}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    examples.append({'prompt':prompt, 'completion':out + tokenizer.eos_token})
train_ds = Dataset.from_list(examples)
print(f'Train: {len(train_ds)} samples')

# Train
peft_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','v_proj','o_proj'])
training_args = SFTConfig(output_dir=S1_OUTPUT_DIR, learning_rate=LR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BS, gradient_accumulation_steps=GAS,
    logging_steps=LOG_STEPS, save_steps=SAVE_STEPS, save_strategy='steps',
    save_only_model=True,
    bf16=torch.cuda.is_available(), fp16=not torch.cuda.is_available(),
    max_length=MAX_SEQ_LEN, packing=False, report_to='none',
    warmup_ratio=0.03, lr_scheduler_type='cosine')
trainer = SFTTrainer(model=model, args=training_args, train_dataset=train_ds,
    processing_class=tokenizer, peft_config=peft_config)
trainer.train()

fd = os.path.join(S1_OUTPUT_DIR, 'final_adapter')
trainer.model.save_pretrained(fd); tokenizer.save_pretrained(fd)
print(f'S1 final: {fd}')
del trainer, model; gc.collect(); torch.cuda.empty_cache()

Failed to load CPU gemm_4bit_forward from kernels-community: No module named 'kernels'. Please make sure you already `pip install kernels` and the kernels >= 0.11.1


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [ ]:
# # ===== Cell 2: Stage-1 Knowledge Injection (train + save) =====
# bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
# if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config,
#     device_map='auto', trust_remote_code=True)
# print(f'CUDA: {torch.cuda.is_available()}')

# # Dataset
# SYS = '/no_think\n你是一个专业的中国法律背诵助手。请严格根据用户要求输出法条内容，不要添加解释。'
# with open(TRAIN_JSON, 'r', encoding='utf-8') as f: raw = json.load(f)
# examples = []
# for item in raw:
#     inst = item.get('instruction','').strip()
#     inp  = item.get('input','').strip()
#     out  = item.get('output','').strip()
#     if not inst or not out: continue
#     msgs = [{'role':'system','content':SYS},{'role':'user','content':inst if not inp else f'{inst}\n\n{inp}'}]
#     prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
#     examples.append({'prompt':prompt, 'completion':out + tokenizer.eos_token})
# train_ds = Dataset.from_list(examples)
# print(f'Train: {len(train_ds)} samples')

# # Train
# peft_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
#     bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','v_proj','o_proj'])
# training_args = SFTConfig(output_dir=S1_OUTPUT_DIR, learning_rate=LR, num_train_epochs=EPOCHS,
#     per_device_train_batch_size=BS, gradient_accumulation_steps=GAS,
#     logging_steps=LOG_STEPS, save_steps=SAVE_STEPS, save_strategy='steps',
#     save_only_model=True,
#     bf16=torch.cuda.is_available(), fp16=not torch.cuda.is_available(),
#     max_length=MAX_SEQ_LEN, packing=False, report_to='none',
#     warmup_ratio=0.03, lr_scheduler_type='cosine')
# trainer = SFTTrainer(model=model, args=training_args, train_dataset=train_ds,
#     processing_class=tokenizer, peft_config=peft_config)
# trainer.train()

# fd = os.path.join(S1_OUTPUT_DIR, 'final_adapter')
# trainer.model.save_pretrained(fd); tokenizer.save_pretrained(fd)
# print(f'S1 final: {fd}')
# del trainer; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ===== 直接跑：格式检查 280+520（不依赖 cell-2）=====
import os, json, re, gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from tqdm.auto import tqdm

MN = '/root/autodl-tmp/employee_code/base_model'
EV = '/root/autodl-tmp/civil_ft/data/eval_civil_code_questions.json'
S1 = '/root/autodl-tmp/civil_ft/checkpoints'

TOP5 = '你是一个专业的中国法律检索专家。\n\n你的任务不是作答，而是为后续数据库检索生成候选法条。\n请根据用户案情，输出最相关的5条法条编号，按相关性从高到低排序。\n\n硬性要求：\n1. 必须恰好输出5条；\n2. 每条都必须是《法律名称》第XXX条；\n3. 不要输出解释、分析、理由、序号或其他任何文字；\n4. 即使不确定，也必须给出5条最可能的候选法条；\n5. 优先输出最核心、最直接适用的法条；\n6. 只输出 JSON，不要输出其他内容。\n\n输出格式：\n{"articles": ["《法律名称》第XXX条", ...]}'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
tok = AutoTokenizer.from_pretrained(MN, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

with open(EV,'r',encoding='utf-8') as f: edat = json.load(f)
tqs = edat[:100]

for ckn in ['checkpoint-280','checkpoint-520']:
    ckp = os.path.join(S1, ckn)
    if not os.path.isdir(ckp): print(f'{ckn}: NOT FOUND'); continue
    print(f'\n=== {ckn} (100q) ===')
    b = AutoModelForCausalLM.from_pretrained(MN, quantization_config=bnb, device_map='auto', trust_remote_code=True)
    m = PeftModel.from_pretrained(b, ckp, is_trainable=False); m.eval()
    jo = ao = c5 = 0
    for it in tqdm(tqs, desc=ckn):
        q = it['question']
        usr = '/no_think\n案情：\n' + q + '\n\n请严格输出 JSON。'
        ms = [{'role':'system','content':TOP5}, {'role':'user','content':usr}]
        tx = tok.apply_chat_template(ms, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        ins = tok(tx, return_tensors='pt').to(m.device)
        ot = m.generate(**ins, max_new_tokens=256, do_sample=False,
            eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
        rw = tok.decode(ot[0][ins['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
        mt = re.search(r'\{.*\}', rw, re.S)
        if mt:
            try:
                obj = json.loads(mt.group()); jo += 1
                if isinstance(obj, dict) and isinstance(obj.get('articles'), list):
                    ao += 1
                    if len(obj['articles']) == 5: c5 += 1
            except: pass
    print(f'  JSON={jo}/100  articles={ao}/100  count5={c5}/100')
    del m, b; gc.collect(); torch.cuda.empty_cache()

print('\n~100/100 -> skip Stage-2. <90 -> need Stage-2')

In [ ]:
# # ===== 格式检查：Stage-1 模型能否正确输出 JSON 格式 =====
# TOP5_SYS = '你是一个专业的中国法律检索专家。\n\n你的任务不是作答，而是为后续数据库检索生成候选法条。\n请根据用户案情，输出最相关的5条法条编号，按相关性从高到低排序。\n\n硬性要求：\n1. 必须恰好输出5条；\n2. 每条都必须是《法律名称》第XXX条；\n3. 不要输出解释、分析、理由、序号或其他任何文字；\n4. 即使不确定，也必须给出5条最可能的候选法条；\n5. 优先输出最核心、最直接适用的法条；\n6. 只输出 JSON，不要输出其他内容。\n\n输出格式：\n{"articles": ["《法律名称》第XXX条", ...]}'

# TEST_QS = [
#     '借钱给朋友，朋友不还怎么办？有录音和转账记录。',
#     '我父亲去世了，银行里有一笔存款，现在要怎么才能取出来？',
#     '我结婚一直和父母在一起住，后房屋拆迁，给我二套房该怎么分。',
# ]

# print('=== Stage-1 Format Check ===')
# model.eval()
# for q in TEST_QS:
#     msgs = [{'role':'system','content':TOP5_SYS},{'role':'user','content':f'/no_think\n案情：\n{q}\n\n请严格输出 JSON。'}]
#     text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
#     inputs = tokenizer(text, return_tensors='pt').to(model.device)
#     outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False,
#         eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
#     raw = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
#     is_json = False; has_arts = False; cnt = 0
#     m = re.search(r'\{.*\}', raw, re.S)
#     if m:
#         try:
#             obj = json.loads(m.group()); is_json = True
#             if isinstance(obj, dict) and isinstance(obj.get('articles'), list):
#                 has_arts = True; cnt = len(obj['articles'])
#         except: pass
#     print(f'\nQ: {q[:60]}...')
#     print(f'  JSON={is_json}  articles={has_arts}  count={cnt}')
#     print(f'  RAW: {raw[:200]}')
# print('\nJSON=True + articles=True + count=5 -> 跳过 Cell 4-6')
# print('否则 -> 需要运行 Stage-2')

In [ ]:
# ===== Cell 4: Stage-2 alignment function =====
def run_stage2(adapter_path, s2_output_dir):
    gc.collect(); torch.cuda.empty_cache()
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True)
    mdl = PeftModel.from_pretrained(base, adapter_path, is_trainable=True)
    args = SFTConfig(output_dir=s2_output_dir, learning_rate=LR_S2, num_train_epochs=1.0,
        per_device_train_batch_size=BS, gradient_accumulation_steps=GAS,
        logging_steps=5, save_strategy='no',
        bf16=torch.cuda.is_available(), fp16=not torch.cuda.is_available(),
        max_length=768, packing=False, report_to='none',
        warmup_ratio=0.05, lr_scheduler_type='cosine',
        completion_only_loss=True)
    t = SFTTrainer(model=mdl, args=args, train_dataset=s2_train, eval_dataset=s2_eval, processing_class=tokenizer)
    t.train()
    final_dir = os.path.join(s2_output_dir, 'final_adapter')
    t.model.save_pretrained(final_dir); tokenizer.save_pretrained(final_dir)
    del t, mdl, base; gc.collect(); torch.cuda.empty_cache()
    return final_dir

In [ ]:
# ===== Cell 5: Eval functions =====
ART_RE = r'《.+?》第[0-9０-９一二三四五六七八九十百千万零〇○两]+条'
CN_M = {'零':0,'〇':0,'○':0,'一':1,'二':2,'两':2,'三':3,'四':4,'五':5,'六':6,'七':7,'八':8,'九':9}
CN_U = {'十':10,'百':100,'千':1000,'万':10000}

def c2i(t):
    if not t: return None
    t=str(t).strip().translate(str.maketrans('０１２３４５６７８９','0123456789'))
    if t.isdigit(): return int(t)
    total=section=number=0
    for ch in t:
        if ch in CN_M: number=CN_M[ch]
        elif ch in CN_U:
            u=CN_U[ch]
            if u==10000: section=(section+(number or 0))*u; total+=section; section=number=0
            else:
                if number==0: number=1
                section+=number*u; number=0
        else: return None
    return total+section+number

def canon(art):
    if not isinstance(art,str): return ''
    x=art.strip(); x=re.sub(r'\s+','',x); x=x.replace('（','(').replace('）',')')
    x=x.translate(str.maketrans('０１２３４５６７８９','0123456789'))
    m=re.match(r'^(《.*?》)第([0-9一二三四五六七八九十百千万零〇○两]+)条$',x)
    if not m: return x
    law=re.sub(r'^《中华人民共和国','《',m.group(1))
    no=c2i(m.group(2))
    return f'{law}第{no}条' if no is not None else f'{law}第{m.group(2)}条'

def extract_articles(text,topk=5):
    mentions=[]
    m=re.search(r'\{.*\}',text,re.S)
    if m:
        try:
            obj=json.loads(m.group())
            if isinstance(obj,dict) and isinstance(obj.get('articles'),list):
                for x in obj['articles']:
                    if isinstance(x,str): mentions.extend(re.findall(ART_RE,x))
        except: pass
    if not mentions: mentions=re.findall(ART_RE,text)[:topk]
    else:
        for m2 in re.findall(ART_RE,text):
            if len(mentions)>=topk: break
            mentions.append(m2)
    return [canon(x) for x in mentions[:topk] if canon(x)]

@torch.no_grad()
def generate_top5(question, mdl):
    msgs=[{'role':'system','content':TOP5_SYSTEM},
          {'role':'user','content':f'/no_think\n案情：\n{question}\n\n请严格输出 JSON。'}]
    text=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    inputs=tokenizer(text,return_tensors='pt').to(mdl.device)
    outputs=mdl.generate(**inputs,max_new_tokens=256,do_sample=True,repetition_penalty=1.05,
        eos_token_id=tokenizer.eos_token_id,pad_token_id=tokenizer.pad_token_id)
    raw=tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:],skip_special_tokens=True).strip()
    return {'raw':raw,'pred':extract_articles(raw)}

with open(EVAL_JSON, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)
print(f'Eval questions: {len(eval_data)}')

def eval_adapter(adapter_path, label):
    gc.collect(); torch.cuda.empty_cache()
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True)
    mdl = PeftModel.from_pretrained(base, adapter_path, is_trainable=False); mdl.eval()
    results=[]; tp_total=pred_total=gt_total=0
    for item in tqdm(eval_data, desc=label):
        gen = generate_top5(item['question'], mdl)
        ps=set(gen['pred']); gs=set(canon(a) for a in item['articles'] if canon(a))
        tp=len(ps&gs); p=tp/max(len(ps),1); r=tp/max(len(gs),1)
        f1=2*p*r/(p+r) if p+r>0 else 0.0
        tp_total+=tp; pred_total+=len(ps); gt_total+=len(gs)
        results.append({'id':item['id'],'question':item['question'],'gold':list(gs),'pred':gen['pred'],
            'P':round(p,6),'R':round(r,6),'F1':round(f1,6),'raw':gen['raw']})
    mp=sum(r['P'] for r in results)/len(results)
    mr=sum(r['R'] for r in results)/len(results)
    mf1=sum(r['F1'] for r in results)/len(results)
    del mdl, base; gc.collect(); torch.cuda.empty_cache()
    s = int(label.split('-')[-1]) if '-' in label else 0
    return {'label':label,'step':s,'P@5':round(mp,4),'R@5':round(mr,4),'F1@5':round(mf1,4),'per_sample':results}

In [ ]:
# ===== Cell 6: Collect checkpoints + Stage-2 align + Eval + Leaderboard =====
targets = []
for name in sorted(os.listdir(S1_OUTPUT_DIR)):
    full = os.path.join(S1_OUTPUT_DIR, name)
    if os.path.isdir(full) and name.startswith('checkpoint-'):
        s = name.split('-')[-1]
        if s.isdigit() and 280 <= int(s) <= 520:
            targets.append((int(s), name, full))
targets = sorted(targets, key=lambda x: x[0])
print(f'Checkpoints (280-520): {len(targets)}')

all_rows = []
for step, label, s1_path in targets:
    print(f'\n{"="*60}')
    print(f'Stage-2 align + Eval: {label}')
    print(f'{"="*60}')

    # Stage-2 alignment
    s2_dir = os.path.join(S2_ROOT_DIR, f'align_{label}')
    os.makedirs(s2_dir, exist_ok=True)
    s2_adapter = run_stage2(s1_path, s2_dir)
    print(f'  Stage-2 done: {s2_adapter}')

    # Evaluate
    row = eval_adapter(s2_adapter, label)
    print(f'  P@5={row["P@5"]:.4f} R@5={row["R@5"]:.4f} F1@5={row["F1@5"]:.4f}')
    all_rows.append({'Checkpoint':label,'Step':step,'P@5':row['P@5'],'R@5':row['R@5'],'F1@5':row['F1@5']})
    with open(os.path.join(RESULTS_DIR, f'{label}.json'), 'w', encoding='utf-8') as f:
        json.dump(row, f, ensure_ascii=False, indent=2)

# Leaderboard
print('\n' + '=' * 60)
print('LEADERBOARD (Stage-2 aligned, 280-520)')
print('=' * 60)
df = pd.DataFrame(all_rows).sort_values('F1@5', ascending=False).reset_index(drop=True)
print(df.to_string(index=False))
best = df.iloc[0]
print(f'\n*** BEST: {best["Checkpoint"]} F1@5={best["F1@5"]:.4f} ***')

with open(os.path.join(RESULTS_DIR, 'leaderboard.json'), 'w', encoding='utf-8') as f:
    json.dump(all_rows, f, ensure_ascii=False, indent=2)
print(f'Saved: {RESULTS_DIR}/')